# Week 39

In [ ]:
!pip install -q --upgrade transformers datasets sacrebleu rouge_score evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 19.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.


In [ ]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
import transformers
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer, DistilBertTokenizerFast
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import os
import evaluate
from transformers import Seq2SeqTrainingArguments
from sacrebleu.metrics import BLEU
from transformers import DataCollatorForSeq2Seq
from datasets import Dataset
from transformers import Seq2SeqTrainer
from google.colab import drive
drive.mount('/content/drive')

## Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])

# Only keep Telugu examples
df_train_te = df_train[df_train['lang'].isin(['te'])]
df_val_te = df_val[df_val['lang'].isin(['te'])]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Drop any rows without an answer
df_train_te = df_train_te[df_train_te['answer_inlang'].notnull()].reset_index(drop=True)
df_val_te = df_val_te[df_val_te['answer_inlang'].notnull()].reset_index(drop=True)

# Combine Telugu question + English context
df_train_te['input_text'] = (
    "question (Telugu): " + df_train_te['question'] +
    " context (English): " + df_train_te['context']
)

df_val_te['input_text'] = (
    "question (Telugu): " + df_val_te['question'] +
    " context (English): " + df_val_te['context']
)

df_train_te['target_text'] = df_train_te['answer_inlang']
df_val_te['target_text'] = df_val_te['answer_inlang']

In [ ]:
# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


## Question + Context

In [ ]:
# Get the question and context seperated by [SEP] token
def get_question_context(row):
    question = row['question']
    context = row['context']
    return question + " " + context

df_train_te['input_text'] = df_train_te.apply(get_question_context, axis=1)
df_val_te['input_text'] = df_val_te.apply(get_question_context, axis=1)

In [ ]:
max_input_length = 768
max_target_length = 64

def tokenize_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=max_input_length, truncation=True)
    labels = tokenizer(examples["target_text"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

train_dataset = Dataset.from_pandas(df_train_te)
val_dataset = Dataset.from_pandas(df_val_te)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
bleu = BLEU()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Load metrics
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.array(predictions)
    if predictions.ndim > 2:
        predictions = np.argmax(predictions, axis=-1)

    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    preds_text = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    em_scores = [int(pred.strip() == ref.strip()) for pred, ref in zip(preds_text, labels_text)]
    exact_match = np.mean(em_scores)

    bleu_score = bleu.corpus_score(preds_text, [labels_text]).score

    rouge_result = rouge_metric.compute(predictions=preds_text, references=labels_text)
    rouge_l = rouge_result["rougeL"]

    chrf_result = chrf_metric.compute(predictions=preds_text, references=labels_text)
    chrf_score = chrf_result["score"]

    return {
        "exact_match": exact_match,
        "bleu": bleu_score,
        "rougeL": rouge_l,
        "chrF": chrf_score
    }


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-te-qa",
    label_smoothing_factor=0.1,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=250,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=50,
    save_safetensors=False
)

In [ ]:
# Train the model

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipython-input-2543715308.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
50,5.480900
100,5.213900
150,4.848200
200,4.576900
250,4.431600
300,4.340900
350,4.154200
400,4.004500
450,3.905700
500,3.922100


TrainOutput(global_step=1750, training_loss=3.6655311192103794, metrics={'train_runtime': 772.5993, 'train_samples_per_second': 16.179, 'train_steps_per_second': 2.265, 'total_flos': 3329929234882560.0, 'train_loss': 3.6655311192103794, 'epoch': 250.0})

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for example in tokenized_val.select(range(20)):
    input_text = example['input_text']
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
    **inputs,
    max_length=64,
    num_beams=5,
    early_stopping=True,
    decoder_start_token_id=model.config.decoder_start_token_id)

    # print("Question + Context:", input_text)
    print("Generated Answer:", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("Reference:", example['target_text'])
    print("-----")

Generated Answer: ఆఫ్రికా సంయుక్త రాష్ట్రాలు
Reference: హన్స్ ఆండర్సాగ్
-----
Generated Answer: ముహమ్మద్ ప్రవక్త
Reference: హరీష్ జైరాజ్
-----
Generated Answer: ఆందోల్
Reference: 1608
-----
Generated Answer: అమెరికా సంయుక్త రాష్ట్రాలు
Reference: మార్చి లేదా ఏప్రిల్
-----
Generated Answer: శ్రీకాకుళం
Reference: కొలంబియా మాదిరిగా అదే పరిమాణం
-----
Generated Answer: అన్నాదురై
Reference: 28,491
-----
Generated Answer: యునైటెడ్ కళాశాల
Reference: 1947
-----
Generated Answer: వెంకటేశ్వరమ్మ
Reference: వెంకటేశ్వర్లు, తల్లి మహాలక్షమ్మ
-----
Generated Answer: ముహమ్మద్ ప్రవక్త
Reference: 87,000
-----
Generated Answer: ముహమ్మద్ అధ్యాపకుడైన
Reference: డివివి దానయ్య
-----
Generated Answer: సర్ ప్రవ
Reference: ఉత్తర ప్రదేశ్
-----
Generated Answer: ప్రధమ
Reference: తాజ్ మహల్
-----
Generated Answer: సిద్ధార్థ సోదరుడైన
Reference: తెలంగాణ లోని మెదక్ జిల్లా ఆందోల్
-----
Generated Answer: అమెరికా సంయుక్త రాష్ట్రాలు
Reference: వెన్నెముక
-----
Generated Answer: సర్ జాన్
Reference: హన్స్ ఆండర్సాగ్
-----
Genera

In [ ]:
# Evaluate the model
results = trainer.evaluate()
print(results)

{'eval_loss': 4.031419277191162, 'eval_exact_match': 0.03, 'eval_bleu': 5.220138067684746, 'eval_rougeL': 0.01, 'eval_chrF': 14.929980913552226, 'eval_runtime': 3.9664, 'eval_samples_per_second': 25.212, 'eval_steps_per_second': 3.278, 'epoch': 250.0}


In [ ]:
# Evaluating answerable and unanswerable
n_train = len(tokenized_train)
n_val = len(tokenized_val)

print(f"Training set size: {n_train}")
print(f"Validation set size: {n_val}")

n_val_answerable = sum(tokenized_val['answerable'])
n_val_unanswerable = n_val - n_val_answerable

print(f"Validation Answerable:   {n_val_answerable}")
print(f"Validation Unanswerable: {n_val_unanswerable}")

n_train_answerable = sum(tokenized_train['answerable'])
n_train_unanswerable = n_train - n_train_answerable

print(f"Training Answerable:   {n_train_answerable}")
print(f"Training Unanswerable: {n_train_unanswerable}")

val_answerable = tokenized_val.filter(lambda x: x["answerable"] == True)
val_unanswerable = tokenized_val.filter(lambda x: x["answerable"] == False)

results_answerable = trainer.evaluate(eval_dataset=val_answerable)
results_unanswerable = trainer.evaluate(eval_dataset=val_unanswerable)

print("Answerable:", results_answerable)
print("Unanswerable:", results_unanswerable)

Training set size: 50
Validation set size: 100
Validation Answerable:   7
Validation Unanswerable: 93
Training Answerable:   5
Training Unanswerable: 45


Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

Answerable: {'eval_loss': 6.403085708618164, 'eval_exact_match': 0.0, 'eval_bleu': 0.0, 'eval_rougeL': 0.0, 'eval_chrF': 4.74610110751149, 'eval_runtime': 0.3522, 'eval_samples_per_second': 19.875, 'eval_steps_per_second': 2.839, 'epoch': 250.0}
Unanswerable: {'eval_loss': 3.7363908290863037, 'eval_exact_match': 0.03225806451612903, 'eval_bleu': 5.370033759169576, 'eval_rougeL': 0.010752688172043012, 'eval_chrF': 15.14295692140786, 'eval_runtime': 4.3648, 'eval_samples_per_second': 21.307, 'eval_steps_per_second': 2.749, 'epoch': 250.0}
